# Boat Party Ticket: V2 causal overlay audit

This notebook is research-only. Candidate D is a frozen Round 1 calendar prior. The teammate's thresholded future-return string is evaluated only as an intentional leakage control; it is never a valid candidate. Valid overlays use only Candidate-D neutral days and causal state through day `t`. Synthetic results retain broad seasonal structure and are generator-conditioned stress tests, not confidence intervals.

In [1]:
import sys
from pathlib import Path
from IPython.display import display

here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / 'trader_interface/data/Boat Party Ticket_price_history.csv').exists())
sys.path.insert(0, str(root / 'research/boat_party'))
from ewma_overlay_audit_v2 import run_v2

v2 = run_v2(root, n_seasonality_paths=150, n_summer_paths=100, seed=20260811)
display(v2['strategy_comparison'][['strategy', 'pnl', 'incremental_pnl_vs_candidate_d', 'max_drawdown', 'active_days', 'trade_count', 'turnover_units', 'max_notional']].round(2))
display(v2['leakage_control'])
display(v2['correctness_checks'][['check', 'passed', 'evidence']])
print('V2 audit complete; outputs written under research/boat_party/results')

,strategy,pnl,incremental_pnl_vs_candidate_d,max_drawdown,active_days,trade_count,turnover_units,max_notional
0,Candidate D,92560.0,0.0,-2460.0,333,92,134000,55390.0
1,D + post EWMA alpha 0.65,101660.0,9100.0,-3080.0,357,72,134000,55390.0
2,D + post EWMA alpha 0.90,99800.0,7240.0,-2460.0,352,73,126000,55390.0
3,D + prior-state EWMA alpha 0.65,100890.0,8330.0,-3080.0,360,72,138000,55390.0
4,D + prior-state EWMA alpha 0.90,101190.0,8630.0,-3080.0,359,72,136000,55390.0
5,D + one-day reversal,89610.0,-2950.0,-3080.0,321,47,92000,55390.0
6,D + causal MA20,76920.0,-15640.0,-4980.0,318,55,104000,55390.0
7,D + Candidate D frozen45,92560.0,0.0,-2460.0,333,92,134000,55390.0
8,D + Summer flat,80010.0,-12550.0,-2460.0,291,65,82000,55390.0
9,D + Summer adaptive EWMA alpha 0.10,93260.0,700.0,-2460.0,333,92,134000,55390.0


,control,signal_length,evaluable_next_return_days,fixed_plus_minus_days,fixed_hit_rate,pnl,incremental_ewma_pnl,included_in_valid_selection,evidence_label
0,thresholded future-return fixed signal,365,364,160,1.0,185880.0,0.0,0,intentional Round 1 leakage control; signal us...
1,thresholded future-return signal plus post EWM...,365,364,160,1.0,196220.0,10340.0,0,intentional Round 1 leakage control; EWMA incr...
2,thresholded future-return signal plus post EWM...,365,364,160,1.0,195810.0,9930.0,0,intentional Round 1 leakage control; EWMA incr...
3,leakage signal reproduces future threshold rule,365,364,160,1.0,185880.0,0.0,0,expected fixed thresholded-return result is ap...


,check,passed,evidence
0,candidate_D_round1_pnl_is_92560,1,pnl=92560.00
1,production_frozen_signal_matches_candidate_D_s...,1,read-only production signal vs frozen Candidat...
2,D + post EWMA alpha 0.65_positions_integral,1,int64
3,D + post EWMA alpha 0.65_positions_within_limit,1,max_abs=1000
4,D + post EWMA alpha 0.65_day_364_flat,1,day364=0
5,D + post EWMA alpha 0.65_pnl_t_to_t_plus_1,1,daily P&L alignment
6,D + prior-state EWMA alpha 0.65_positions_inte...,1,int64
7,D + prior-state EWMA alpha 0.65_positions_with...,1,max_abs=1000
8,D + prior-state EWMA alpha 0.65_day_364_flat,1,day364=0
9,D + prior-state EWMA alpha 0.65_pnl_t_to_t_plus_1,1,daily P&L alignment


V2 audit complete; outputs written under research/boat_party/results


## Key V2 stress summaries

In [2]:
display(v2['seasonality_stress_summary'].round(2))
display(v2['summer_stress_summary'][v2['summer_stress_summary']['scenario_scope'] == 'pooled'].round(2))
display(v2['summer_stress_paired'][v2['summer_stress_paired']['target_equilibrium'].isna()].round(2))
display(v2['portfolio_capital'].round(2))

,strategy,n_paths,median_pnl,p10_pnl,worst_pnl,positive_path_rate,median_max_drawdown,evidence_label
0,Candidate D,1100,55413.51,39046.55,17720.00,1.0,-1834.76,generator-conditioned seasonality-preserving s...
1,D + one-day reversal,1100,53217.86,38030.36,6390.48,1.0,-2073.21,generator-conditioned seasonality-preserving s...
2,D + post EWMA alpha 0.65,1100,57786.90,40834.67,18379.76,1.0,-2000.77,generator-conditioned seasonality-preserving s...
3,D + prior-state EWMA alpha 0.65,1100,57786.90,40834.67,18379.76,1.0,-2000.77,generator-conditioned seasonality-preserving s...


,scenario_scope,target_equilibrium,transition_mode,transition_days,strategy,n_paths,median_full_pnl,p10_full_pnl,worst_full_pnl,median_summer_pnl,p10_summer_pnl,worst_summer_pnl,median_max_drawdown,positive_path_rate,evidence_label
182,pooled,NaN,NaN,NaN,Candidate D frozen45,2600,60126.67,41818.71,21650.00,9193.10,-1181.05,-5739.52,-7200.48,1.0,generator-conditioned summer-equilibrium stres...
183,pooled,NaN,NaN,NaN,Summer adaptive EWMA alpha 0.10,2600,64217.14,47822.14,26280.00,13739.76,5245.90,-4330.48,-7121.43,1.0,generator-conditioned summer-equilibrium stres...
184,pooled,NaN,NaN,NaN,Summer flat,2600,52221.90,37471.29,25067.14,0.00,0.00,0.00,-7041.90,1.0,generator-conditioned summer-equilibrium stres...
185,pooled,NaN,NaN,NaN,"Summer guarded=10,2,0.10",2600,60754.52,42741.90,21650.00,9450.95,7.57,-4678.10,-7200.48,1.0,generator-conditioned summer-equilibrium stres...
186,pooled,NaN,NaN,NaN,Summer shrunk weight=0.50,2600,62403.10,44573.38,21650.00,11371.67,838.71,-4612.86,-7200.48,1.0,generator-conditioned summer-equilibrium stres...
187,pooled,NaN,NaN,NaN,Summer shrunk weight=0.75,2600,61189.05,42804.00,21650.00,10307.14,-813.48,-6382.38,-7200.48,1.0,generator-conditioned summer-equilibrium stres...
188,pooled,NaN,NaN,NaN,Summer shrunk weight=0.90,2600,60464.05,42311.57,21650.00,9471.90,-1087.48,-5739.52,-7200.48,1.0,generator-conditioned summer-equilibrium stres...


,target_equilibrium,transition_mode,transition_days,strategy,n_paths,paired_median_full_difference,paired_p10_full_difference,paired_worst_full_difference,paired_median_summer_difference,paired_p10_summer_difference,paired_worst_summer_difference,evidence_label
156,NaN,NaN,NaN,Summer adaptive EWMA alpha 0.10,2600,3568.57,-3760.38,-18486.67,3568.57,-3760.38,-18486.67,paired generator-conditioned summer stress; no...
157,NaN,NaN,NaN,Summer flat,2600,-9193.10,-19898.71,-30535.71,-9193.10,-19898.71,-30535.71,paired generator-conditioned summer stress; no...
158,NaN,NaN,NaN,"Summer guarded=10,2,0.10",2600,0.00,0.00,-18101.90,0.00,0.00,-18101.90,paired generator-conditioned summer stress; no...
159,NaN,NaN,NaN,Summer shrunk weight=0.50,2600,1016.67,-1985.71,-9435.24,1016.67,-1985.71,-9435.24,paired generator-conditioned summer stress; no...
160,NaN,NaN,NaN,Summer shrunk weight=0.75,2600,0.00,-1068.00,-7017.14,0.00,-1068.00,-7017.14,paired generator-conditioned summer stress; no...
161,NaN,NaN,NaN,Summer shrunk weight=0.90,2600,0.00,-41.14,-4962.86,0.00,-41.14,-4962.86,paired generator-conditioned summer stress; no...


,strategy,max_boat_notional,max_other_instrument_capital,max_combined_capital,combined_budget_violation_days,max_incremental_boat_capital_vs_candidate_d,mean_incremental_boat_capital_active_days,overlap_days_with_other_positions,overlapping_other_instruments,evidence_label
0,Candidate D,55390.0,528631.0,575991.0,0,0.0,0.00,333,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
1,D + post EWMA alpha 0.65,55390.0,528631.0,575991.0,0,54820.0,3182.80,357,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
2,D + post EWMA alpha 0.90,55390.0,528631.0,575991.0,0,54820.0,2570.48,352,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
3,D + prior-state EWMA alpha 0.65,55390.0,528631.0,575991.0,0,54820.0,3545.78,360,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
4,D + prior-state EWMA alpha 0.90,55390.0,528631.0,575991.0,0,54820.0,3443.84,359,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
5,D + one-day reversal,55390.0,528631.0,575991.0,0,54820.0,4395.42,321,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
6,D + causal MA20,55390.0,528631.0,575991.0,0,54820.0,4014.09,318,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
7,D + Candidate D frozen45,55390.0,528631.0,575991.0,0,0.0,0.00,333,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
8,D + Summer flat,55390.0,528631.0,575991.0,0,0.0,0.00,291,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
9,D + Summer adaptive EWMA alpha 0.10,55390.0,528631.0,575991.0,0,0.0,0.00,333,Bread;Fintech Token;MenuDash;Sausage Sizzle;Th...,read-only current production other-instrument ...
